In [ ]:
import sys
from pathlib import Path

_HERE = Path.cwd() if '__file__' not in globals() else Path(__file__).resolve().parent
_ROOT = _HERE.parents[1]
sys.path.insert(0, str(_HERE))

import numpy as np
import mujoco
import mujoco.viewer
from core import FixedBaseRobotState, Mujoco_Wrapper, Mujoco_Kernel
from utils import SimScheduler, Visualizer, DataLogger, plot_helpers
from control import TrajectoryGenerator

%load_ext autoreload
%autoreload 2

# Panda 7-DOF arm + 2-finger gripper
# Pinocchio 없이 MuJoCo API만 사용
MJCF = str(_ROOT / 'models' / 'scenes' / 'panda' / 'scene_torque.xml')

# MuJoCo body/site 이름
PANDA_FRAMES = {
    "ee":   "grasp_site",
    "base": "link0",
}

PANDA_JOINT_NAMES = [
    'joint1', 'joint2', 'joint3', 'joint4',
    'joint5', 'joint6', 'joint7',
]

GRIPPER_ACTUATOR_IDX = 7
GRIPPER_OPEN  = 255
GRIPPER_CLOSE = 0

### SO(3) 유틸 — pin.log3 대체
Pinocchio 없이 순수 numpy로 rotation error 계산

In [ ]:
def log3(R):
    """SO(3) → so(3) : rotation matrix → axis-angle vector (3,)."""
    cos_a = np.clip((np.trace(R) - 1.0) / 2.0, -1.0, 1.0)
    angle = np.arccos(cos_a)
    if abs(angle) < 1e-10:
        return np.zeros(3)
    return angle / (2.0 * np.sin(angle)) * np.array([
        R[2, 1] - R[1, 2],
        R[0, 2] - R[2, 0],
        R[1, 0] - R[0, 1],
    ])

### TaskSpaceController (MuJoCo 전용 — pinocchio 의존 없음)

In [ ]:
class TaskSpaceController:
    def __init__(self, wrapper: Mujoco_Wrapper):
        self.wrapper = wrapper

        wn_pos = 20.0
        wn_ori = 20.0
        self.Kp_pos = wn_pos**2 * np.eye(3)
        self.Kp_ori = wn_ori**2 * np.eye(3)
        self.Kd_lin = 2 * wn_pos * np.eye(3)
        self.Kd_ang = 2 * wn_ori * np.eye(3)

        self.I        = np.eye(6)
        self.damping2 = 1e-3 ** 2

        self.pose_err  = np.zeros(6)
        self.twist_err = np.zeros(6)
        self.F_des     = np.zeros(6)

    def compute_error(self, pos, R, twist, pos_des, R_des, twist_des):
        self.pose_err[:3] = pos_des - pos
        self.pose_err[3:] = log3(R_des @ R.T)   # pin.log3 대체
        self.twist_err[:] = twist_des - twist
        return self.pose_err, self.twist_err

    def compute_impedance_torque(self, pose_err, twist_err, J, M_inv, nle):
        Lambda_inv = J @ M_inv @ J.T + self.damping2 * self.I
        Lambda     = np.linalg.inv(Lambda_inv)

        self.F_des[:3] = self.Kp_pos @ pose_err[:3] + self.Kd_lin @ twist_err[:3]
        self.F_des[3:] = self.Kp_ori @ pose_err[3:] + self.Kd_ang @ twist_err[3:]

        return J.T @ (Lambda @ self.F_des) + nle

### State container + Orchestrator

In [ ]:
class PandaState(FixedBaseRobotState):
    """Panda — 9-DOF (7 arm + 2 finger). MJCF q/dq 크기와 일치."""
    NQ = 9
    NV = 9

    def __init__(self):
        self._q  = np.zeros(self.NQ)
        self._dq = np.zeros(self.NV)
        self.arm_pos    = self._q[0:7]
        self.arm_vel    = self._dq[0:7]
        self.finger_pos = self._q[7:9]
        self.finger_vel = self._dq[7:9]


class RobotOrchestrator:
    """Panda impedance — MuJoCo_Wrapper 전용.
    kernel(9-DOF) → wrapper(7-DOF arm): qpos/qvel[:7] 슬라이싱.
    """

    def __init__(self, kernel, wrapper, controller, traj, state, logger=None):
        self.kernel     = kernel
        self.wrapper    = wrapper
        self.controller = controller
        self.traj       = traj
        self.state      = state
        self.logger     = logger

        self.target_pos   = np.zeros(3)
        self.target_R     = np.eye(3)
        self.target_twist = np.zeros(6)

        self.current_ee_pos = np.zeros(3)
        self.current_ee_R   = np.eye(3)
        self.current_twist  = np.zeros(6)

        self.ctrl_tau = np.zeros(wrapper.nv)   # 7-DOF arm

    def on_control(self, t):
        # 1. 상태 갱신 — wrapper는 kernel.data를 참조하므로 qpos/qvel만 동기화
        self.kernel.update_robot_state(self.state)
        self.wrapper.update_model(self.state.q[:7], self.state.dq[:7])

        # 2. 궤적 평가
        interpolated_oM_ee, target_twist_new = self.traj.evaluate(t)
        self.target_pos[:]   = interpolated_oM_ee.translation
        self.target_R[:]     = interpolated_oM_ee.rotation
        self.target_twist[:] = target_twist_new

        # 3. 현재 EE 상태
        oMf, v_lin, v_ang = self.wrapper.compute_ee_state_W("ee")
        self.current_ee_pos[:]  = oMf.translation
        self.current_ee_R[:]    = oMf.rotation
        self.current_twist[:3]  = v_lin
        self.current_twist[3:]  = v_ang

        # 4. 오차 + 임피던스 토크
        pose_err, twist_err = self.controller.compute_error(
            self.current_ee_pos, self.current_ee_R, self.current_twist,
            self.target_pos,     self.target_R,     self.target_twist
        )
        self.ctrl_tau[:] = self.controller.compute_impedance_torque(
            pose_err, twist_err,
            self.wrapper.compute_J_W("ee"),
            self.wrapper.M_inv,
            self.wrapper.nle,
        )
        self.kernel.ctrl_tau = self.ctrl_tau
        self.kernel.data.ctrl[GRIPPER_ACTUATOR_IDX] = GRIPPER_OPEN

        # 5. 로깅
        if self.logger is not None:
            self.logger.log(
                times      = t,
                target_pos = self.target_pos,
                actual_pos = self.current_ee_pos,
                target_vel = self.target_twist[:3],
                actual_vel = self.current_twist[:3],
                cmd_tau    = self.ctrl_tau,
                act_tau    = self.kernel.data.qfrc_actuator[:7],
                q          = self.kernel.data.qpos[:7],
                dq         = self.kernel.data.qvel[:7],
                ddq        = self.kernel.data.qacc[:7],
            )

    def on_render(self, t, visualizer):
        visualizer.draw_axes(self.current_ee_pos, self.current_ee_R, size=0.2)
        visualizer.draw_axes(self.target_pos,     self.target_R,     size=0.2)

### 시스템 초기화

In [ ]:
state  = PandaState()
kernel = Mujoco_Kernel(MJCF, joint_names_pin_order=PANDA_JOINT_NAMES)

# wrapper 생성 전에 keyframe으로 reset — wrapper __init__이 현재 qpos를 읽음
kernel.reset_to_keyframe()
wrapper = Mujoco_Wrapper(kernel, frames=PANDA_FRAMES)

init_ee_pos = wrapper._p_ee_W.copy()
init_ee_R   = wrapper._R_ee_W.copy()

controller = TaskSpaceController(wrapper)

print(f'wrapper.nv = {wrapper.nv}  (7-DOF arm)')
print(f'wrapper.m  = {wrapper.m:.3f} kg')
print(f'kernel.model.nq = {kernel.model.nq}, nu = {kernel.model.nu}')
print(f'초기 EE 위치: {init_ee_pos}')
print(f'초기 EE z축: {init_ee_R[:, 2]}')

### 궤적 설정

In [ ]:
wp_pos = [
    init_ee_pos,
    init_ee_pos + np.array([ 0.10,  0.15,  0.00]),
    init_ee_pos + np.array([-0.10,  0.15, -0.10]),
    init_ee_pos + np.array([-0.10, -0.15, -0.10]),
    init_ee_pos + np.array([ 0.10, -0.15,  0.00]),
    init_ee_pos,
]
wp_rot = [init_ee_R] * len(wp_pos)

traj = TrajectoryGenerator(wp_list=wp_pos, R_list=wp_rot, target_speed=0.2)
print(f'Waypoints: {len(wp_pos)}개, target_speed=0.2 m/s')

### Main Loop

In [ ]:
kernel.reset_to_keyframe()
logger = DataLogger()

orch  = RobotOrchestrator(kernel, wrapper, controller, traj, state, logger)
sched = SimScheduler(kernel.model, kernel.data, ctrl_hz=500, render_hz=60)
sched.run(on_control=orch.on_control, on_render=orch.on_render)

### 결과 시각화

In [ ]:
times      = logger.get('times')
target_pos = logger.get('target_pos')
actual_pos = logger.get('actual_pos')
target_vel = logger.get('target_vel')
actual_vel = logger.get('actual_vel')
q          = logger.get('q')
dq         = logger.get('dq')
ddq        = logger.get('ddq')

plot_helpers.plot_3d_trajectory(
    pos_log=actual_pos, waypoints=wp_pos, title="Panda EE 3D Trajectory (MuJoCo only)"
)
plot_helpers.plot_ee_tracking(
    times=times, desired=target_pos, actual=actual_pos
)
plot_helpers.plot_velocity_tracking(
    times=times, des_vel=target_vel, act_vel=actual_vel,
    title="Panda EE Linear Velocity"
)
plot_helpers.plot_joint_state(
    times=times, q=q, dq=dq, ddq=ddq,
    joint_names=PANDA_JOINT_NAMES,
    title="Panda Joint States (arm 7-DOF)"
)

err = target_pos - actual_pos
plot_helpers.summarize_tracking_error(
    times, err,
    axes=('X', 'Y', 'Z'),
    plot_axes=('X', 'Y', 'Z'),
    title="Panda Position Error (MuJoCo only)",
)